# 01 — Connect Brokerage Accounts (Plaid Sandbox)

This notebook connects to Plaid's Sandbox environment and creates simulated brokerage account connections.

In Sandbox mode, we skip the browser-based Plaid Link flow and create tokens directly via the API.
When you move to Development/Production, you'll use Plaid Link to authenticate with your real brokerages.

**Prerequisites:**
- `.env` file with your Plaid credentials (copy from `.env.example`)
- Dependencies installed: `uv sync`

## Setup

In [ ]:
import json
import os
import sys

# Allow imports from the project root
sys.path.insert(0, os.path.abspath(".."))

from plaid_config import get_plaid_client, PLAID_ENV

client = get_plaid_client()
print(f"✅ Plaid client initialized (environment: {PLAID_ENV})")

## Create Sandbox Connections

In Sandbox, we simulate connecting to a brokerage by calling `/sandbox/public_token/create` directly.
This gives us a `public_token` that we exchange for a persistent `access_token` — just like the real flow,
but without needing a browser.

We'll create 4 simulated connections to mirror your real setup:
- **stash** (bulk of portfolio)
- **robinhood**
- **sofi**
- **fidelity**

> Note: In Sandbox, all connections use the same simulated institution. The nicknames are just labels
> so our code is ready for real accounts later.

In [ ]:
from plaid.model.sandbox_public_token_create_request import SandboxPublicTokenCreateRequest
from plaid.model.item_public_token_exchange_request import ItemPublicTokenExchangeRequest
from plaid.model.products import Products

# Sandbox institution that supports investments
# ins_115616 = "Vanguard" in Plaid's sandbox
SANDBOX_INSTITUTION_ID = "ins_115616"

# The nicknames for each simulated brokerage
BROKERAGES = ["stash", "robinhood", "sofi", "fidelity"]

access_tokens = {}

for name in BROKERAGES:
    print(f"Connecting '{name}'...", end=" ")
    
    # Step 1: Create a sandbox public token
    sandbox_request = SandboxPublicTokenCreateRequest(
        institution_id=SANDBOX_INSTITUTION_ID,
        initial_products=[Products("investments")],
    )
    sandbox_response = client.sandbox_public_token_create(sandbox_request)
    public_token = sandbox_response.public_token
    
    # Step 2: Exchange for a persistent access token
    exchange_request = ItemPublicTokenExchangeRequest(public_token=public_token)
    exchange_response = client.item_public_token_exchange(exchange_request)
    
    access_tokens[name] = exchange_response.access_token
    print(f"✅ Connected (item_id: {exchange_response.item_id[:12]}...)")

print(f"\n🎉 All {len(access_tokens)} brokerages connected!")

## Save Access Tokens

We save the tokens to `access_tokens.json` (gitignored) so the next notebook can load them
without reconnecting.

In [ ]:
TOKENS_FILE = os.path.join("..", "access_tokens.json")

with open(TOKENS_FILE, "w") as f:
    json.dump(access_tokens, f, indent=2)

print(f"💾 Saved {len(access_tokens)} access tokens to {TOKENS_FILE}")
print(f"   Brokerages: {list(access_tokens.keys())}")
print(f"\n⚠️  This file is gitignored — never commit access tokens.")

## Quick Verification

Let's make sure one of the tokens works by pulling account info.

In [ ]:
from plaid.model.investments_holdings_get_request import InvestmentsHoldingsGetRequest

# Test with the first token
test_name = list(access_tokens.keys())[0]
test_token = access_tokens[test_name]

response = client.investments_holdings_get(
    InvestmentsHoldingsGetRequest(access_token=test_token)
)

print(f"✅ Test pull from '{test_name}':")
print(f"   Accounts: {len(response.accounts)}")
print(f"   Holdings: {len(response.holdings)}")
print(f"   Securities: {len(response.securities)}")
print(f"\n👉 Run notebook 02 to see your full portfolio.")